# GLS map-making of a `litebird_sim` simulation with pysanepic

This notebook simulates one day of LiteBIRD observations with `litebird_sim`
(four LFT 140 GHz detectors, realistic scanning strategy, no HWP, CMB sky,
1/f noise with the IMO parameters), then makes maps with:

1. the `litebird_sim` **binner** (no noise model),
2. the `litebird_sim` **destriper**,
3. **pysanepic** GLS, which uses the full 1/f noise spectrum.

All maps are in Galactic coordinates (the `litebird_sim` default) and are compared with the binned map of
the noiseless signal, so the residuals contain only noise.

Requirements: `pysanepic` and a `litebird_sim` version providing
`Simulation.make_sanepic_gls_map` (until [litebird_sim#568](https://github.com/litebird/litebird_sim/pull/568)
is merged: `pip install git+https://github.com/litebird/litebird_sim@sanepic_gls`).
The simulation takes about a minute on a laptop.

In [ ]:
import time

import astropy.time
import healpy as hp
import litebird_sim as lbs
import matplotlib.pyplot as plt
import numpy as np
from pysanepic import GLSParameters

## Parameters

In [ ]:
telescope, channel = "LFT", "L4-140"
# Two T/B pairs, Q-type and U-type: without HWP both are needed to separate Q and U
detlist = ["000_001_016_QA_140_T", "000_001_016_QA_140_B", "000_001_014_UA_140_T", "000_001_014_UA_140_B"]
duration_days = 1.0
nside = 64

# pysanepic applies N^-1 independently on chunks of data: 1 hour here.
chunk_s = 3600.0
# Destriper baseline length (the lbs default of 100 samples is far too short here)
baseline_s = 600.0

## Simulation with `litebird_sim`

In [ ]:
imo = lbs.Imo(flatfile_location=lbs.PTEP_IMO_LOCATION)
sim = lbs.Simulation(
    base_path="lbs_example_output",
    imo=imo,
    start_time=astropy.time.Time("2030-01-01T00:00:00"),
    duration_s=duration_days * 86400,
    random_seed=12345,
)
sim.set_instrument(
    lbs.InstrumentInfo.from_imo(imo, f"/releases/vPTEP/satellite/{telescope}/instrument_info")
)
dets = [
    lbs.DetectorInfo.from_imo(
        url=f"/releases/vPTEP/satellite/{telescope}/{channel}/{d}/detector_info", imo=imo
    )
    for d in detlist
]
sim.set_scanning_strategy(imo_url="/releases/vPTEP/satellite/scanning_parameters/")
sim.create_observations(detectors=dets)
sim.prepare_pointings()  # no HWP: the case GLS map-making is designed for

ch = lbs.FreqChannelInfo.from_imo(
    url=f"/releases/vPTEP/satellite/{telescope}/{channel}/channel_info", imo=imo
)
sky = sim.get_sky(
    parameters=lbs.SkyGenerationParams(
        make_cmb=True, make_fg=False, seed_cmb=1, apply_beam=True,
        units="K_CMB", output_type="map", nside=nside,
    ),
    channels=[ch],
    store_in_observation=True,
)
sim.fill_tods()

obs = sim.observations[0]
signal = obs.tod.copy()  # kept to build the noiseless reference map
sim.add_noise(noise_type="one_over_f")

fs = obs.sampling_rate_hz
print(f"{obs.n_detectors} detectors x {obs.n_samples} samples at {fs} Hz")
print(f"fknee = {obs.fknee_mhz} mHz, alpha = {obs.alpha}, NET = {obs.net_ukrts} uK sqrt(s)")

## GLS map-making with pysanepic

`Simulation.make_sanepic_gls_map` passes pointings, HWP angle, TOD and the 1/f
noise parameters (NET, f_knee, α, f_min) of each detector to pysanepic, like
the other `litebird_sim` map-makers. N^-1 is applied on chunks of `chunk_s` seconds.

In [ ]:
t0 = time.time()
gls = sim.make_sanepic_gls_map(nside=nside, params=GLSParameters(chunk_s=chunk_s))
print(f"{time.time() - t0:.1f} s, {gls.iterations} iterations, converged: {gls.converged}")
gls_map = np.where(gls.maps == hp.UNSEEN, np.nan, gls.maps)

plt.semilogy(gls.residuals)
plt.xlabel("iteration")
plt.ylabel(r"$|r|^2 / |b|^2$")
plt.title("PCG convergence");

## Binner and destriper from `litebird_sim`, and the noiseless reference

In [ ]:
binned = sim.make_binned_map(nside=nside).binned_map
destriped = sim.make_destriped_map(
    nside=nside,
    params=lbs.DestriperParameters(
        samples_per_baseline=int(baseline_s * fs),
        threshold=1e-12,
        iter_max=500,
    ),
).destriped_map

noisy = obs.tod.copy()
obs.tod[:] = signal
reference = sim.make_binned_map(nside=nside).binned_map
obs.tod[:] = noisy

## Comparison

Residual = map − noiseless reference. The I monopole is removed, since none of
the map-makers constrains it. The white-noise floor is σ²/hits per pixel,
i.e. what an ideal map-maker would reach if there were no 1/f noise.

In [ ]:
seen = np.isfinite(gls_map[0]) & np.isfinite(gls_map[1])
residuals = {}
for name, mp in [("binned", binned), ("destriped", destriped), ("pysanepic GLS", gls_map)]:
    r = np.where(seen, mp - reference, np.nan)
    r[0] -= np.nanmean(r[0])
    residuals[name] = r

hits = gls.hit_map
sigma_k = obs.net_ukrts[0] * np.sqrt(fs) / 1e6
print(f"white-noise floor, I: {1e6 * np.sqrt(np.mean(sigma_k**2 / hits[seen])):.2f} uK")
print(f"{'':15s}   rms residual I / Q / U [uK]")
for name, r in residuals.items():
    print(f"{name:15s}   " + " / ".join(f"{1e6 * np.nanstd(x):6.2f}" for x in r))

In [ ]:
plt.figure(figsize=(15, 8))
for i, (name, r) in enumerate(residuals.items()):
    for j, comp in enumerate("IQ"):
        hp.mollview(1e6 * r[j], sub=(2, 3, 3 * j + i + 1), title=f"{name}: {comp} residual",
                    unit="uK", min=-100, max=100, cmap="RdBu_r", badcolor="lightgray")

Without HWP, 1/f noise affects intensity and polarization alike. On this day of
data (rms residuals I/Q/U: binned 19.0/26.8/27.2 µK, destriped 17.2/24.9/25.1 µK,
GLS 16.6/24.0/24.2 µK, white-noise floor in I 15.0 µK) the GLS map is the
closest to the floor in all three components. The destriper approaches it only
with long baselines; the differences grow with longer simulations and higher
f_knee.